In [15]:
! pip install pandas

  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)


In [13]:
import faiss

# Check if Faiss can see the GPU
res = faiss.StandardGpuResources()
print(f"Faiss GPU Resources initialized: {res}")

# Check if a GPU index can be created
index = faiss.IndexFlatL2(128) # CPU index
gpu_index = faiss.index_cpu_to_gpu(res, 0, index)
print("Successfully moved index to GPU")

Faiss GPU Resources initialized: <faiss.swigfaiss.StandardGpuResources; proxy of <Swig Object of type 'faiss::gpu::StandardGpuResources *' at 0x74d75ab52030> >
Successfully moved index to GPU


In [16]:
import pandas as pd
data = [['Where are your headquarters located?', 'location'],
['Throw my cellphone in the water', 'random'],
['Network Access Control?', 'networking'],
['Address', 'location']]
df = pd.DataFrame(data, columns = ['text', 'category'])

from sentence_transformers import SentenceTransformer
text = df['text']
encoder = SentenceTransformer('all-MiniLM-L6-v2')
vectors = encoder.encode(text)

/home/faciraci/RAGAnalysis/FederatedRAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 895.54it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
from sentence_transformers import SentenceTransformer
text = df['text']
encoder = SentenceTransformer("paraphrase-mpnet-base-v2")
vectors = encoder.encode(text)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 849.28it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/paraphrase-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
import faiss

vector_dimension = vectors.shape[1]
index = faiss.IndexFlatL2(vector_dimension)
faiss.normalize_L2(vectors)
index.add(vectors)

In [19]:
import numpy as np

search_text = 'where is your office?'
search_vector = encoder.encode(search_text)
_vector = np.array([search_vector])
faiss.normalize_L2(_vector)

In [20]:
k = index.ntotal
distances, ann = index.search(_vector, k=k)

In [21]:
results = pd.DataFrame({'distances': distances[0], 'ann': ann[0]})

In [22]:
results.head()

,distances,ann
0,0.584873,0
1,1.175950,3
2,1.644266,2
3,1.919768,1
